In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# **Oil Sands Analysis Project**

# Goal: 
The idea of the project is to get an idea of Alberta oil production vs WTI (West-Texas intermediate) prices. On paper, it seems very simple: if global oil prices are higher, Alberta should produce more. The issue with this idea is that many factors are in play when trying to increase oil production in Alberta. These can be weather, taxes, emissions, and so on and so forth.    

### Data Sources
Our sources come from the St. Louis Federal Reserve and the Government of Alberta website

In [2]:
#The St. Louis Fed data is separated by date and price
#The Alberta data set is separated by non vs. conventional oil. Non-conventional is MOSTLY oilsands products (bitumen and synthetic)
#We will first do all of the cleaning on the St. Louis Fed data and then clean the Alberta data after.
data = pd.read_csv("data/DCOILWTICO.csv") 
data_alb = pd.read_csv("data/alberta_production_by_type.csv")

In [3]:
data.head() 

,observation_date,DCOILWTICO
0,1986-01-02,25.56
1,1986-01-03,26.00
2,1986-01-06,26.53
3,1986-01-07,25.85
4,1986-01-08,25.87


### Data Cleaning: FRED data

In [4]:
data.shape

(10513, 2)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10513 entries, 0 to 10512
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  10513 non-null  object 
 1   DCOILWTICO        10143 non-null  float64
dtypes: float64(1), object(1)
memory usage: 164.4+ KB


In [6]:
data.columns = ["date", "price"]

#Convert to date time so we can do analysis later
data["date"] = pd.to_datetime(data["date"])
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10513 entries, 0 to 10512
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    10513 non-null  datetime64[ns]
 1   price   10143 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 164.4 KB


In [7]:
data.isnull().sum()

date       0
price    370
dtype: int64

In [8]:
#Some of the data is null as seen above. 370 data points is a very small portion so we will front and backfill
data["price"] = data["price"].ffill().bfill()

In [9]:
data.isnull().sum()

date     0
price    0
dtype: int64

In [10]:
data.describe()

,date,price
count,10513,10513.000000
mean,2006-02-24 19:11:56.712641536,48.326903
min,1986-01-02 00:00:00,-36.980000
25%,1996-01-30 00:00:00,20.340000
50%,2006-02-24 00:00:00,43.210000
75%,2016-03-23 00:00:00,71.280000
max,2026-04-20 00:00:00,145.310000
std,NaN,29.448196


In [11]:
#Since our Alberta data only starts from 2007, we dont want data points from before that
data = data[data["date"] >= "2007-01-01"]

In [12]:
data.head()

,date,price
5477,2007-01-01,60.85
5478,2007-01-02,60.77
5479,2007-01-03,58.31
5480,2007-01-04,55.65
5481,2007-01-05,56.29


In [13]:
data.to_csv("data/wti_cleaned.csv", index=False)

### Data Cleaning: Alberta Data

In [14]:
data_alb.columns = ["date", "produced", "type", "labels"]
data_alb["date"] = pd.to_datetime(data_alb["date"])

In [15]:
data_alb.head()

,date,produced,type,labels
0,2007-01-01,2707743.4,Conventional Oil,2007-01-01T00:00:00
1,2007-02-01,2450594.2,Conventional Oil,2007-02-01T00:00:00
2,2007-03-01,2703836.8,Conventional Oil,2007-03-01T00:00:00
3,2007-04-01,2567241.4,Conventional Oil,2007-04-01T00:00:00
4,2007-05-01,2614545.2,Conventional Oil,2007-05-01T00:00:00


In [16]:
#THe labels column is just another date column, simply omit
data_alb = data_alb.drop(columns=["labels"])

In [17]:
data_alb.head()

,date,produced,type
0,2007-01-01,2707743.4,Conventional Oil
1,2007-02-01,2450594.2,Conventional Oil
2,2007-03-01,2703836.8,Conventional Oil
3,2007-04-01,2567241.4,Conventional Oil
4,2007-05-01,2614545.2,Conventional Oil


In [18]:
data_alb.isnull().sum()

date        0
produced    0
type        0
dtype: int64

In [19]:
data_alb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      460 non-null    datetime64[ns]
 1   produced  460 non-null    float64       
 2   type      460 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 10.9+ KB


In [20]:
#We should have exactly 230 for each type of oil
data_alb["type"].value_counts()

type
Conventional Oil        230
Non-Conventional Oil    230
Name: count, dtype: int64

In [21]:
#Now we do a pivot so the conventional/non-conventional are side by side and not merged into one column
alb_wide = data_alb.pivot(index="date", columns="type", values="produced")
alb_wide = alb_wide.reset_index()
alb_wide.head()

type,date,Conventional Oil,Non-Conventional Oil
0,2007-01-01,2707743.4,5604533.2
1,2007-02-01,2450594.2,5414892.8
2,2007-03-01,2703836.8,6082110.5
3,2007-04-01,2567241.4,5463210.8
4,2007-05-01,2614545.2,5460600.2


### Saving our cleaned data as a csv

In [22]:
data.to_csv("data/wti_cleaned.csv", index=False)
data_alb.to_csv("data/alberta_production_long.csv", index=False)
alb_wide.to_csv("data/alberta_production_wide.csv", index=False)